# Markov Chain Text Generator

A simple Markov-chain based text generator (supports order-1 and higher-order models).

**Usage:**
1. Run the implementation cell to get `build_markov_chain` and `generate_text`.
2. Train on a text corpus (e.g., `Readme.md` or your own file) to build the chain.
3. Use `generate_text(chain, order=<n>, length=<tokens>)` to produce output.

---

*This cell explains the approach and provides a quick demo in the following cells.*

In [1]:
from collections import defaultdict, deque
import random
import json
import os

# --- Tokenization ----------------------------------------------------------------

def tokenize(text):
    """Very small tokenizer: splits on whitespace. Replace with a better one if needed."""
    return text.split()


# --- Build the chain -------------------------------------------------------------

def build_markov_chain(tokens, order=1):
    """Builds an n-th order Markov chain.

    Args:
        tokens: iterable of tokens (strings)
        order: number of prior tokens to condition on

    Returns:
        dict mapping tuple(context) -> list(of next-token choices)
    """
    if order < 1:
        raise ValueError("order must be >= 1")
    chain = defaultdict(list)
    dq = deque(maxlen=order)
    for tok in tokens:
        if len(dq) == order:
            chain[tuple(dq)].append(tok)
        dq.append(tok)
    return dict(chain)


# --- Generate text ----------------------------------------------------------------

def generate_text(chain, order=1, length=50, seed=None):
    """Generate text (as tokens joined by spaces) from a chain.

    Args:
        chain: dict returned by build_markov_chain
        order: order used when chain was built
        length: number of tokens to generate (not counting seed tokens)
        seed: optional seed context (tuple, list, or whitespace string)
    """
    if not chain:
        return ""

    # Normalize seed
    if seed is None:
        seed = random.choice(list(chain.keys()))
    else:
        if isinstance(seed, str):
            seed = tuple(seed.split())
        else:
            seed = tuple(seed)

    # If seed doesn't match order, pick a random key
    if len(seed) != order:
        seed = random.choice(list(chain.keys()))

    out = list(seed)
    for _ in range(length):
        key = tuple(out[-order:])
        choices = chain.get(key)
        if not choices:
            break
        out.append(random.choice(choices))
    return " ".join(out)


# --- Save / Load helpers ---------------------------------------------------------

def save_chain(chain, path):
    """Save chain to a JSON file. Keys are joined with a control separator."""
    sep = "\x1f"  # rarely used ASCII unit separator
    serial = {sep.join(k): v for k, v in chain.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serial, f, ensure_ascii=False,indent=2)


def load_chain(path):
    """Load chain saved by save_chain."""
    sep = "\x1f"
    with open(path, "r", encoding="utf-8") as f:
        serial = json.load(f)
    return {tuple(k.split(sep)): v for k, v in serial.items()}


# --- Small utility: pretty-printing sample continuations -------------------------

def sample_continuations(chain, order=1, n=3, length=30):
    samples = []
    for _ in range(n):
        samples.append(generate_text(chain, order=order, length=length))
    return samples

In [2]:
# --- Demo: train on Readme.md (if available) and show a few examples ---

sample_path = "Readme.md"
text = ""
if os.path.exists(sample_path):
    with open(sample_path, encoding="utf-8") as f:
        text = f.read()
# If file exists but is empty, fall back to a short sample corpus
if not text or not text.strip():
    text = (
        "Markov chains are simple models for sequence generation. "
        "This small demo falls back to this short string if no Readme is found or it is empty."
    )

tokens = tokenize(text)
print("Tokens in corpus:", len(tokens))

# Build a 2nd-order model (change order to experiment)
order = 2
chain = build_markov_chain(tokens, order=order)
print("Number of distinct states in chain:", len(chain))

# Show a few generated samples
for i, s in enumerate(sample_continuations(chain, order=order, n=5, length=40), 1):
    print(f"\nSample {i}:\n", s)

# (Optional) Save chain for later use
save_path = "markov_chain.json"
save_chain(chain, save_path)
print("Saved chain to:", save_path)


Tokens in corpus: 28
Number of distinct states in chain: 26

Sample 1:
 statistics and programming learning systems improve with experience

Sample 2:
 networks data science combines statistics and programming learning systems improve with experience

Sample 3:
 learning uses neural networks data science combines statistics and programming learning systems improve with experience

Sample 4:
 programming learning systems improve with experience

Sample 5:
 systems improve with experience
Saved chain to: markov_chain.json




### What this file is

It’s a **Markov Chain text generator** notebook.
It learns patterns from text and then **generates fake text** that looks similar.

Nothing more.

---

### Big picture (1 line)

👉 **Input text → learn word transitions → generate new text**

---

### What each part does

#### 1. Markdown cell (title)

Just explains the goal:

* Build a Markov model
* Support order-1, order-2, etc.
* Generate text

No code logic here.

---

#### 2. Imports (first code cell)

```python
from collections import defaultdict, deque
import random
import json
import os
```

Why:

* `defaultdict` → store transitions easily
* `deque` → sliding window of last words
* `random` → pick next word randomly
* `os` → check if a file exists

Nothing fancy.

---

#### 3. `tokenize(text)`

```python
def tokenize(text):
```

What it does:

* Takes raw text
* Splits it into words/tokens
* Very simple tokenizer (not NLP-grade)

👉 Output:
`"hello world"` → `["hello", "world"]`

---

#### 4. `build_markov_chain(tokens, order)`

Core logic.

What it learns:

* For **order = 1**
  looks at **1 word → next word**
* For **order = 2**
  looks at **2 words → next word**

Example (order = 1):

```
"I love AI"
I → love
love → AI
```

Stored internally like:

```python
{
  ("I",): ["love"],
  ("love",): ["AI"]
}
```

This is the **Markov chain**.

---

#### 5. `generate_text(chain, start, length)`

What it does:

* Start with a word (or words)
* Repeatedly:

  * look at last `order` words
  * randomly choose a next word from learned options
* Stops after `length` words

This is why output feels random but “text-like”.

---

#### 6. Demo cell (last code cell)

```python
sample_path = "Readme.md"
```

What happens:

* If `Readme.md` exists:

  * read it
  * train the Markov model
  * print generated text
* If it doesn’t exist:

  * nothing useful happens

👉 If you didn’t have `Readme.md`, **this explains your confusion**.

---

### Why it feels confusing

Blunt truth:

* No comments explaining Markov chains
* Assumes you already know:

  * tokens
  * n-grams
  * probability sampling
* Demo depends on an **external file**

Bad beginner UX.

---

### One-sentence mental model

> The code learns “what word usually comes after what” and then plays that game randomly.

---


